In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances
from sklearn.ensemble import RandomForestRegressor
import xarray as xr
from hbv_module import HBV
import umap.umap_ as umap
from sklearn.svm import SVR

# -------------------------
# Efficiency metrics
# -------------------------
def nse(sim, obs):
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)

def kge(sim, obs):
    r = np.corrcoef(sim, obs)[0, 1]
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)
    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)

# -------------------------
# Load catchment attributes
# -------------------------
df_attrs = pd.read_csv("../Clustering/GB_ATTRIBUTES/catchment_selection.csv", index_col=0)
df_attrs = df_attrs.dropna()
df_attrs.index = df_attrs.index.astype(str)

# -------------------------
# Load calibrated parameters
# -------------------------
param_file = "../calib_params/best_NSE_cal.csv"
df_params = pd.read_csv(param_file)
df_params["catchment_id"] = df_params["catchment_id"].astype(str)
df_params.set_index("catchment_id", inplace=True)

# Keep only common catchments
common_catchments = df_attrs.index.intersection(df_params.index)
if len(common_catchments) == 0:
    raise ValueError("No catchments left after aligning attributes and parameters.")

df_attrs = df_attrs.loc[common_catchments]
df_params = df_params.loc[common_catchments]

# -------------------------
# Standardize attributes
# -------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_attrs.values)

# -------------------------
# LOO-CV Regionalization Setup
# -------------------------
catchment_folder = "../data"
output_dir = "SVM_REGRESSION_6_clusters_hbv_regionalization_NSE_VAL"
os.makedirs(output_dir, exist_ok=True)
sim_output_dir = os.path.join(output_dir, "simulated_flows")
os.makedirs(sim_output_dir, exist_ok=True)

start_date = "2009-01-01"
end_date = "2013-12-31"

hbv_param_cols = [
    "parBETA","parFC","parK0","parK1","parK2","parLP","parPERC","parUZL",
    "parTT","parCFMAX","parCFR","parCWH","paralpha","parbeta"
]

methods = ["nearest", "weighted", "regression"]
topK = 3  # for weighted method
n_clusters = 6  # number of clusters in k-means

# -------------------------
# Main Loop: run all methods (leak-free UMAP + k-means)
# -------------------------
for transfer_method in methods:
    print(f"\n▶ Running leak-free LOO-CV with transfer method: {transfer_method}\n")
    results = []

    for fold_target in df_attrs.index:
        # -------------------------
        # Define donors (exclude target)
        # -------------------------
        donors = df_attrs.index.drop(fold_target)
        target_index = df_attrs.index.get_loc(fold_target)
        target_attr = X_scaled[target_index].reshape(1, -1)
        donor_indices = [df_attrs.index.get_loc(c) for c in donors]
        donor_scaled = X_scaled[donor_indices]

        # -------------------------
        # Fit UMAP on donors only
        # -------------------------
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        donor_umap = reducer.fit_transform(donor_scaled)
        target_umap = reducer.transform(target_attr)

        # -------------------------
        # Fit k-means on donor UMAP only
        # -------------------------
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        donor_clusters = kmeans.fit_predict(donor_umap)
        target_cluster = kmeans.predict(target_umap)[0]

        # -------------------------
        # Select donor catchments in same cluster
        # -------------------------
        cluster_donors = [donors[i] for i, c in enumerate(donor_clusters) if c == target_cluster]
        if len(cluster_donors) == 0:
            cluster_donors = donors  # fallback if no donors in same cluster

        donor_attrs = donor_umap[[i for i, c in enumerate(donor_clusters) if donors[i] in cluster_donors]]

        # -------------------------
        # Select donor parameters
        # -------------------------
        if transfer_method == "nearest":
            dists = pairwise_distances(target_umap, donor_attrs).flatten()
            nearest_idx = np.argmin(dists)
            donor_id = cluster_donors[nearest_idx]
            donor_params = df_params.loc[donor_id, hbv_param_cols].values
            precip_corr = df_params.loc[donor_id, "parprecip_corr_factor"]

        elif transfer_method == "weighted":
            dists = pairwise_distances(target_umap, donor_attrs).flatten()
            topk_idx = np.argsort(dists)[:min(topK, len(cluster_donors))]
            weights = 1 / (dists[topk_idx] + 1e-6)
            weights /= weights.sum()
            topk_catchments = [cluster_donors[i] for i in topk_idx]
            donor_params_matrix = df_params.loc[topk_catchments, hbv_param_cols].values
            donor_params = np.average(donor_params_matrix, axis=0, weights=weights)
            precip_corr = np.average(df_params.loc[topk_catchments, "parprecip_corr_factor"].values, weights=weights)
            donor_id = ",".join(topk_catchments)

        elif transfer_method == "regression":
            donor_X = donor_attrs
            donor_Y = df_params.loc[cluster_donors, hbv_param_cols].values
            donor_params = []
            for i in range(donor_Y.shape[1]):
                svr = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
                svr.fit(donor_X, donor_Y[:, i])
                pred = svr.predict(target_umap)
                donor_params.append(pred[0])
            donor_params = np.array(donor_params)
            precip_corr = df_params.loc[cluster_donors[0], "parprecip_corr_factor"]
            donor_id = "regression"

        # -------------------------
        # Load target catchment data
        # -------------------------
        nc_file = os.path.join(catchment_folder, f"{fold_target}.nc")
        if not os.path.exists(nc_file):
            results.append({"target": fold_target, "donor": donor_id, "cluster": target_cluster, "NSE": np.nan, "KGE": np.nan})
            continue

        ds = xr.open_dataset(nc_file)
        df = ds.to_dataframe().reset_index()
        df["date"] = pd.to_datetime(df.get("time", df.get("date")))
        df = df[(df["date"] >= start_date) & (df["date"] <= end_date)]
        if df.empty:
            results.append({"target": fold_target, "donor": donor_id, "cluster": target_cluster, "NSE": np.nan, "KGE": np.nan})
            continue

        input_data = df[["total_precipitation_sum", "potential_evaporation_sum", "temperature"]].values
        obs_flow = df["discharge_spec"].values.flatten()
        input_data[:, 0] *= precip_corr

        # -------------------------
        # Run HBV
        # -------------------------
        try:
            hbv = HBV()
            discharge, _ = hbv.run_model(input_data, donor_params)
            discharge = discharge.flatten()

            score_nse = nse(discharge, obs_flow)
            score_kge = kge(discharge, obs_flow)

            flow_df = pd.DataFrame({
                "date": df["date"].values,
                "observed": obs_flow,
                "simulated": discharge,
                "cluster": target_cluster,
                "method": transfer_method,
                "donor": donor_id
            })
            flow_file = os.path.join(sim_output_dir, f"{transfer_method}_{fold_target}_flows.csv")
            flow_df.to_csv(flow_file, index=False)

        except Exception as e:
            print(f"⚠️ Error for {fold_target} ({transfer_method}): {e}")
            score_nse, score_kge = np.nan, np.nan

        results.append({
            "target": fold_target,
            "donor": donor_id,
            "cluster": target_cluster,
            "NSE": round(score_nse, 3) if not np.isnan(score_nse) else np.nan,
            "KGE": round(score_kge, 3) if not np.isnan(score_kge) else np.nan
        })

    # -------------------------
    # Save results
    # -------------------------
    df_results = pd.DataFrame(results)
    df_results.to_csv(os.path.join(output_dir, f"LOOCV_results_{transfer_method}.csv"), index=False)

    # Cluster-level metrics
    cluster_metrics = []
    for cluster_id in range(n_clusters):
        df_cluster = df_results[df_results["cluster"] == cluster_id]
        cluster_metrics.append({
            "cluster": cluster_id,
            "num_catchments": len(df_cluster),
            "mean_NSE": round(df_cluster["NSE"].dropna().mean(), 3) if not df_cluster["NSE"].dropna().empty else np.nan,
            "mean_KGE": round(df_cluster["KGE"].dropna().mean(), 3) if not df_cluster["KGE"].dropna().empty else np.nan
        })
    pd.DataFrame(cluster_metrics).to_csv(
        os.path.join(output_dir, f"cluster_metrics_{transfer_method}.csv"), index=False
    )

    # Overall metrics
    overall_nse = df_results["NSE"].dropna().mean()
    overall_kge = df_results["KGE"].dropna().mean()
    pd.DataFrame([{
        "transfer_method": transfer_method,
        "overall_NSE": round(overall_nse, 3) if not np.isnan(overall_nse) else np.nan,
        "overall_KGE": round(overall_kge, 3) if not np.isnan(overall_kge) else np.nan
    }]).to_csv(os.path.join(output_dir, f"overall_metrics_{transfer_method}.csv"), index=False)

    print(f"✅ Finished {transfer_method}. Results saved.")